# Document structure and IIIF with gallica-sdk

This notebook exercises the typed structural and IIIF primitives used by the SDK: Pagination, Toc, IIIF Presentation and IIIF Image info.json. The ARKs are the same public examples used by the live validation suite.

In [ ]:
from gallica import Gallica

with Gallica() as gallica:
    document = gallica.document('bpt6k5738219s')
    metadata = document.metadata()
    pagination = document.pagination()

assert metadata.raw_xml
assert pagination.image_views > 0
print('title:', metadata.record.title)
print('image views:', pagination.image_views)
print('has TOC:', pagination.has_toc)
print('TOC location:', pagination.toc_location)
print('first logical pages:', [(page.order, page.number) for page in pagination.pages[:5]])

In [ ]:
with Gallica() as gallica:
    toc = gallica.document('bpt6k97540464').toc()

assert toc.format in {'html', 'tei'}
assert toc.raw
print('TOC format:', toc.format)
print('well formed:', toc.well_formed)
print('payload prefix:', toc.raw[:160])

In [ ]:
import httpx

from gallica import GallicaResponseError

manifest = None
with Gallica() as gallica:
    try:
        manifest = gallica.document('btv1b550076223').iiif_manifest()
    except httpx.HTTPStatusError as exc:
        if exc.response.status_code != 403:
            raise
        print('IIIF Presentation is environment-limited from this runner (HTTP 403).')
    except GallicaResponseError as exc:
        if 'IIIF Presentation manifest returned HTML' not in str(exc):
            raise
        print('IIIF Presentation is environment-limited from this runner (HTML response).')

    info = gallica.document('btv1b53066668g').page(1).iiif_info()

if manifest is not None:
    assert manifest.version == '2'
    assert manifest.canvas_count is not None and manifest.canvas_count > 0
    print('Presentation version:', manifest.version)
    print('canvas count:', manifest.canvas_count)

assert info.width > 0 and info.height > 0
print('Image API detected version:', info.version)
print('image dimensions:', info.width, info.height)
print('info.json prefix:', info.raw_json[:160])

The distinction between the two IIIF observations is intentional. Gallica's tested Presentation manifest self-identifies as Presentation 2, while the tested Image `info.json` currently lacks the v2/v3 markers used by the SDK and is therefore reported as `unknown`. The SDK preserves this uncertainty instead of importing a protocol version from external documentation.